In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

numero_registros = 1000000

# 2. Gerar o DataFrame distribuído nativamente com PySpark
df_raw = spark.range(0, numero_registros).withColumnRenamed("id", "contrato_id")

df_bronze = df_raw \
    .withColumn("departamento_id", (F.rand() * 5 + 1).cast(IntegerType())) \
    .withColumn("fornecedor_id", (F.rand() * 100 + 1).cast(IntegerType())) \
    .withColumn("valor_contrato", F.round(F.rand() * 50000 + 1000, 2)) \
    .withColumn("data_assinatura", F.expr("date_add(current_date(), - cast(rand() * 365 as int))")) \
    .withColumn("status", F.expr("case when rand() > 0.8 then 'CANCELADO' " +
                                 "when rand() > 0.2 then 'ATIVO' " +
                                 "else NULL end")) # Injetando valores nulos para a limpeza futura

df_bronze = df_bronze.withColumn("departamento_id", 
                                 F.when(F.rand() > 0.95, F.lit(None)).otherwise(F.col("departamento_id")))

df_duplicados = df_bronze.sample(fraction=0.05)
df_bronze_final = df_bronze.union(df_duplicados)

df_bronze_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_contratos")

print(f"Camada Bronze gravada com sucesso! Total de registros brutos: {df_bronze_final.count()}")

Camada Bronze gravada com sucesso! Total de registros brutos: 1050575


In [0]:
# 1. Ler os dados diretamente da tabela gerenciada na Bronze
df_bronze_lido = spark.table("bronze_contratos")

# 2. Limpeza de Dados (Data Cleansing)
df_silver = df_bronze_lido \
    .dropDuplicates(["contrato_id"]) \
    .fillna({"status": "DESCONHECIDO"}) \
    .fillna({"departamento_id": 999}) # ID padrão para não informados

# 3. Enriquecimento de Dados: Simulando estrutura de um Tribunal/Órgão Público
df_silver = df_silver.withColumn(
    "nome_departamento",
    F.when(F.col("departamento_id") == 1, "Diretoria de TI")
     .when(F.col("departamento_id") == 2, "Infraestrutura e Obras")
     .when(F.col("departamento_id") == 3, "Vara Cível")
     .when(F.col("departamento_id") == 4, "Vara Criminal")
     .when(F.col("departamento_id") == 5, "Recursos Humanos")
     .otherwise("Não Informado")
)

df_silver = df_silver \
    .withColumn("ano", F.year("data_assinatura")) \
    .withColumn("mes", F.month("data_assinatura"))

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("ano", "mes") \
    .saveAsTable("silver_contratos")

print(f"Camada Silver gravada. Total de registros limpos e únicos: {df_silver.count()}")

Camada Silver gravada. Total de registros limpos e únicos: 1000000


In [0]:
from pyspark.sql.window import Window

# 1. Ler dados da tabela Silver
df_silver_lido = spark.table("silver_contratos")

# Focar apenas nos contratos ativos para a contabilidade
df_ativos = df_silver_lido.filter(F.col("status") == "ATIVO")

# 2. Análise 1: Agregação de Gastos por Departamento
df_gold_gastos = df_ativos \
    .groupBy("nome_departamento") \
    .agg(
        F.sum("valor_contrato").alias("custo_total"),
        F.count("contrato_id").alias("quantidade_contratos")
    ) \
    .orderBy(F.desc("custo_total"))

window_spec = Window.partitionBy("nome_departamento")

df_gold_anomalias = df_ativos \
    .withColumn("media_departamento", F.avg("valor_contrato").over(window_spec)) \
    .withColumn("variacao_percentual", 
                ((F.col("valor_contrato") - F.col("media_departamento")) / F.col("media_departamento")) * 100) \
    .filter(F.col("variacao_percentual") > 50) # Retorna contratos com valor 50% acima da média do seu setor

# 4. Salvar os resultados na Camada Gold como tabelas gerenciadas
df_gold_gastos.write.format("delta").mode("overwrite").saveAsTable("gold_gastos_departamento")
df_gold_anomalias.write.format("delta").mode("overwrite").saveAsTable("gold_contratos_anomalos")

print("Camada Gold processada com sucesso!")

Camada Gold processada com sucesso!


In [0]:
%sql
-- Identificando as maiores anomalias de contratos por departamento
SELECT 
    nome_departamento, 
    contrato_id, 
    ROUND(valor_contrato, 2) AS valor_contrato, 
    ROUND(media_departamento, 2) AS media_setor,
    ROUND(variacao_percentual, 2) AS percentual_acima_media
FROM gold_contratos_anomalos 
ORDER BY percentual_acima_media DESC
LIMIT 15;

nome_departamento,contrato_id,valor_contrato,media_setor,percentual_acima_media
Diretoria de TI,788706,50997.94,25916.86,96.78
Diretoria de TI,756242,50999.24,25916.86,96.78
Diretoria de TI,40701,50999.6,25916.86,96.78
Diretoria de TI,921537,50998.33,25916.86,96.78
Diretoria de TI,449446,50999.46,25916.86,96.78
Diretoria de TI,671627,50998.45,25916.86,96.78
Diretoria de TI,632883,50998.53,25916.86,96.78
Diretoria de TI,559538,50999.67,25916.86,96.78
Diretoria de TI,46603,50998.56,25916.86,96.78
Diretoria de TI,506121,50995.38,25916.86,96.77
